# Module 3 • Classical Natural Language Processing

# Lesson 16 • Classical Text Classification with Naive Bayes, Logistic Regression, and Support Vector Machines

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner to Intermediate  
**Estimated study time:** 100–130 minutes

---

## Scope

This lesson develops leakage-safe classical text-classification pipelines using
Multinomial Naive Bayes, Logistic Regression, and Linear Support Vector
Machines. It covers train-test splitting, cross-validation, evaluation
metrics, confusion matrices, class imbalance, feature interpretation,
hyperparameter tuning, and multilingual considerations.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define text classification and identify common applications;
- prepare labeled text data for supervised learning;
- build leakage-safe pipelines with TF-IDF features;
- explain the assumptions of Multinomial Naive Bayes;
- interpret Logistic Regression coefficients;
- explain the maximum-margin idea behind linear SVMs;
- compare accuracy, precision, recall, F1, and macro F1;
- interpret confusion matrices and classification reports;
- use stratified train-test splitting and cross-validation;
- address class imbalance with suitable metrics and class weights;
- tune model and vectorizer hyperparameters safely;
- perform error analysis and inspect influential features.

## Table of Contents

1. What Is Text Classification?
2. Supervised Learning Workflow
3. Example Dataset
4. Train-Test Splitting
5. Leakage-Safe Pipelines
6. Multinomial Naive Bayes
7. Logistic Regression
8. Linear Support Vector Machines
9. Predictions and Decision Scores
10. Evaluation Metrics
11. Confusion Matrices
12. Classification Reports
13. Cross-Validation
14. Model Comparison
15. Class Imbalance
16. Hyperparameter Tuning
17. Feature Interpretation
18. Error Analysis
19. Multilingual and Arabic Considerations
20. Reproducibility and Deployment
21. Knowledge Check
22. Exercises
23. Summary and Next Lesson

# 1. What Is Text Classification?

**Text classification** assigns one or more labels to a text.

Examples include:

- sentiment classification;
- spam detection;
- topic classification;
- intent detection;
- toxicity detection;
- language identification;
- document routing;
- urgency classification.

In [ ]:
import pandas as pd

classification_tasks = pd.DataFrame(
    [
        ("Sentiment", "positive / negative / neutral"),
        ("Spam detection", "spam / not spam"),
        ("Topic classification", "sports / politics / technology"),
        ("Intent detection", "billing / technical / account"),
        ("Urgency detection", "urgent / routine"),
    ],
    columns=["Task", "Example labels"],
)

classification_tasks

Classical text classifiers typically use sparse features such as counts,
TF-IDF, word n-grams, or character n-grams.

# 2. Supervised Learning Workflow

A standard workflow is:

```text
Collect labeled data
    ↓
Inspect and clean data
    ↓
Split into training and test sets
    ↓
Fit preprocessing and model on training data
    ↓
Evaluate on unseen data
    ↓
Perform error analysis
    ↓
Revise features, model, or data
```

> **Key Idea**
>
> The test set must remain unseen until final evaluation. Preprocessing that
> learns from data must be fitted only on training data.

# 3. Example Dataset

This notebook uses a small synthetic customer-support dataset with three
classes:

- `account`;
- `billing`;
- `technical`.

In [ ]:
data = pd.DataFrame(
    [
        ("I cannot reset my password", "account"),
        ("My login code never arrives", "account"),
        ("The account verification failed", "account"),
        ("I need to change my email address", "account"),
        ("My profile is locked", "account"),
        ("I cannot access my account", "account"),
        ("How can I update my password?", "account"),
        ("The sign-in page rejects my credentials", "account"),
        ("I was charged twice this month", "billing"),
        ("Please send me the latest invoice", "billing"),
        ("I need a refund for the last payment", "billing"),
        ("Why did my subscription price increase?", "billing"),
        ("The payment was declined", "billing"),
        ("My credit card was charged incorrectly", "billing"),
        ("Where can I download the receipt?", "billing"),
        ("Please cancel the paid plan", "billing"),
        ("The application crashes after startup", "technical"),
        ("The page loads very slowly", "technical"),
        ("The integration stopped working", "technical"),
        ("I receive an unexpected server error", "technical"),
        ("The mobile app freezes repeatedly", "technical"),
        ("The upload feature does not work", "technical"),
        ("The dashboard shows a blank screen", "technical"),
        ("Notifications are not being delivered", "technical"),
    ],
    columns=["text", "label"],
)

data.head()

In [ ]:
print("Dataset size:", len(data))
print("\nClass distribution:")
print(data["label"].value_counts())

The dataset is intentionally small and balanced. Its purpose is to demonstrate
workflow and interpretation, not to establish production performance.

# 4. Train-Test Splitting

A train-test split separates model development data from final evaluation data.

**Stratification** preserves class proportions in both partitions.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.25,
    random_state=42,
    stratify=data["label"],
)

print("Training size:", len(X_train))
print("Test size:", len(X_test))
print("\nTraining labels:")
print(y_train.value_counts())
print("\nTest labels:")
print(y_test.value_counts())

Without stratification, a small test set may contain too few examples from one
class.

# 5. Leakage-Safe Pipelines

A pipeline combines vectorization and classification.

This ensures that the vectorizer vocabulary and IDF values are fitted only on
the training portion used in each split or cross-validation fold.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

base_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
)

Unsafe workflow:

```text
fit vectorizer on complete dataset
split transformed matrix
evaluate model
```

Safe workflow:

```text
split raw text
fit pipeline on training text
transform test text using fitted pipeline
```

# 6. Multinomial Naive Bayes

**Multinomial Naive Bayes** estimates the probability of a class from feature
frequencies.

Its simplifying assumption is that features are conditionally independent
given the class.

This assumption is usually unrealistic for language, but the classifier often
performs well with sparse text features.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            MultinomialNB(alpha=1.0),
        ),
    ]
)

nb_pipeline.fit(X_train, y_train)
nb_predictions = nb_pipeline.predict(X_test)

print(nb_predictions)

`alpha` controls additive smoothing. Smoothing prevents unseen features from
forcing probabilities to zero.

# 7. Logistic Regression

**Logistic Regression** learns a linear decision boundary and estimates class
probabilities.

For multiclass text classification, one linear weight is learned for each
class-feature combination.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

lr_pipeline.fit(X_train, y_train)
lr_predictions = lr_pipeline.predict(X_test)

print(lr_predictions)

Logistic Regression supports:

- regularization;
- class weighting;
- probability estimates;
- interpretable linear coefficients.

# 8. Linear Support Vector Machines

A linear Support Vector Machine finds a decision boundary with a large margin
between classes.

Linear SVMs are often strong baselines for high-dimensional sparse text data.

In [ ]:
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LinearSVC(
                C=1.0,
                random_state=42,
            ),
        ),
    ]
)

svm_pipeline.fit(X_train, y_train)
svm_predictions = svm_pipeline.predict(X_test)

print(svm_predictions)

`C` controls the trade-off between margin size and training errors.

A standard `LinearSVC` does not directly provide calibrated probabilities.

# 9. Predictions and Decision Scores

Models can return:

- predicted labels;
- probabilities;
- decision scores.

Probability outputs should not be assumed to be perfectly calibrated.

In [ ]:
sample_texts = [
    "I need a copy of my invoice",
    "The app crashes when I upload a file",
    "I forgot my login password",
]

sample_predictions = pd.DataFrame(
    {
        "Text": sample_texts,
        "Naive Bayes": nb_pipeline.predict(sample_texts),
        "Logistic Regression": lr_pipeline.predict(sample_texts),
        "Linear SVM": svm_pipeline.predict(sample_texts),
    }
)

sample_predictions

In [ ]:
lr_probabilities = lr_pipeline.predict_proba(sample_texts)

probability_frame = pd.DataFrame(
    lr_probabilities,
    columns=lr_pipeline.named_steps["classifier"].classes_,
    index=sample_texts,
)

probability_frame.round(3)

A high predicted probability does not guarantee correctness. Calibration and
uncertainty evaluation require additional analysis.

# 10. Evaluation Metrics

For a class:

- **Precision:** how many predicted positives were correct;
- **Recall:** how many actual positives were found;
- **F1:** harmonic mean of precision and recall.

\[
Precision = \frac{TP}{TP + FP}
\]

\[
Recall = \frac{TP}{TP + FN}
\]

\[
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
\]

**Accuracy** measures the fraction of all correct predictions.

Accuracy can be misleading when classes are imbalanced.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

accuracy = accuracy_score(y_test, lr_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    lr_predictions,
    average="macro",
    zero_division=0,
)

print(f"Accuracy:        {accuracy:.3f}")
print(f"Macro precision: {precision:.3f}")
print(f"Macro recall:    {recall:.3f}")
print(f"Macro F1:        {f1:.3f}")

Macro averaging gives equal importance to each class. Weighted averaging gives
more influence to larger classes.

# 11. Confusion Matrices

A confusion matrix shows actual labels against predicted labels.

In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(data["label"].unique())

lr_confusion = confusion_matrix(
    y_test,
    lr_predictions,
    labels=labels,
)

pd.DataFrame(
    lr_confusion,
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

Off-diagonal cells identify which classes are confused with each other.

# 12. Classification Reports

A classification report provides per-class precision, recall, F1, and support.

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        lr_predictions,
        zero_division=0,
    )
)

Always inspect per-class metrics. A strong average may hide one weak class.

# 13. Cross-Validation

Cross-validation evaluates a model across several training-validation splits.

**Stratified k-fold cross-validation** preserves class proportions in each
fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)

lr_cv_scores = cross_val_score(
    lr_pipeline,
    data["text"],
    data["label"],
    cv=cv,
    scoring="f1_macro",
)

print("Logistic Regression macro F1:", lr_cv_scores.round(3))
print("Mean:", lr_cv_scores.mean().round(3))
print("Standard deviation:", lr_cv_scores.std().round(3))

Reporting both mean and standard deviation provides more information than one
score.

# 14. Model Comparison

Compare models under the same folds and metric.

In [ ]:
models = {
    "Multinomial Naive Bayes": nb_pipeline,
    "Logistic Regression": lr_pipeline,
    "Linear SVM": svm_pipeline,
}

comparison_rows = []

for name, pipeline in models.items():
    scores = cross_val_score(
        pipeline,
        data["text"],
        data["label"],
        cv=cv,
        scoring="f1_macro",
    )

    comparison_rows.append(
        {
            "Model": name,
            "Mean macro F1": scores.mean(),
            "Std macro F1": scores.std(),
        }
    )

model_comparison = pd.DataFrame(comparison_rows)
model_comparison.sort_values(
    "Mean macro F1",
    ascending=False,
).round(3)

Model selection should also consider training time, inference time,
interpretability, probability requirements, and deployment constraints.

# 15. Class Imbalance

A class-imbalanced dataset contains substantially different numbers of
examples per class.

Problems may include:

- majority-class bias;
- weak minority recall;
- misleading accuracy;
- unstable evaluation.

In [ ]:
imbalanced_data = pd.DataFrame(
    [
        *[("routine message", "routine") for _ in range(18)],
        *[("urgent service failure", "urgent") for _ in range(3)],
    ],
    columns=["text", "label"],
)

imbalanced_data["label"].value_counts()

Possible strategies include:

- macro F1;
- class weights;
- resampling;
- threshold adjustment;
- collecting more minority examples;
- stratified splitting.

In [ ]:
balanced_lr_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

balanced_lr_pipeline

Class weighting changes the training objective but does not create new
information. Minority-class quality still depends on representative examples.

# 16. Hyperparameter Tuning

Hyperparameters include:

- n-gram range;
- `min_df`;
- TF-IDF normalization;
- Naive Bayes `alpha`;
- Logistic Regression regularization;
- SVM `C`;
- class weights.

In [ ]:
from sklearn.model_selection import GridSearchCV

tuning_pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

parameter_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.5, 1.0, 2.0],
}

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=parameter_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=None,
)

grid_search.fit(
    data["text"],
    data["label"],
)

print("Best parameters:", grid_search.best_params_)
print("Best macro F1:", round(grid_search.best_score_, 3))

Hyperparameters must be selected using training or validation data, not the
final test set.

# 17. Feature Interpretation

Logistic Regression and linear SVMs expose feature coefficients.

In [ ]:
lr_full = lr_pipeline.fit(
    data["text"],
    data["label"],
)

feature_names = (
    lr_full
    .named_steps["tfidf"]
    .get_feature_names_out()
)

classifier = lr_full.named_steps["classifier"]
class_names = classifier.classes_

top_features = {}

for class_index, class_name in enumerate(class_names):
    coefficients = classifier.coef_[class_index]
    top_indices = coefficients.argsort()[-8:][::-1]

    top_features[class_name] = [
        feature_names[index]
        for index in top_indices
    ]

pd.DataFrame(
    dict(
        [
            (class_name, pd.Series(features))
            for class_name, features in top_features.items()
        ]
    )
)

Feature weights reflect this dataset and preprocessing pipeline. They are not
universal semantic truths.

# 18. Error Analysis

Error analysis should inspect:

- the text;
- actual label;
- predicted label;
- model confidence or score;
- influential features;
- ambiguity;
- annotation quality;
- missing domain examples.

In [ ]:
error_frame = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual": y_test.reset_index(drop=True),
        "predicted": pd.Series(lr_predictions),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame

In [ ]:
error_frame[~error_frame["correct"]]

Common error categories include:

- ambiguous wording;
- missing context;
- label overlap;
- rare vocabulary;
- negation;
- spelling variation;
- insufficient training examples;
- annotation inconsistency;
- domain shift.

# 19. Multilingual and Arabic Considerations

Classical multilingual classification requires attention to:

- language identification;
- script;
- tokenization;
- normalization;
- morphology;
- stop words;
- code-switching;
- class balance by language;
- domain coverage.

## 19.1 Arabic Classification

Arabic classifiers may compare:

- word TF-IDF;
- character TF-IDF;
- light stemming;
- clitic segmentation;
- word and character feature unions.

Character n-grams can provide robustness to spelling and morphology, but they
do not replace linguistic analysis.

In [ ]:
arabic_texts = [
    "لا أستطيع تسجيل الدخول",
    "نسيت كلمة المرور",
    "أريد نسخة من الفاتورة",
    "تم خصم المبلغ مرتين",
    "التطبيق يتوقف عن العمل",
    "الصفحة لا تفتح",
]

arabic_labels = [
    "account",
    "account",
    "billing",
    "billing",
    "technical",
    "technical",
]

arabic_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char",
                ngram_range=(2, 5),
            ),
        ),
        (
            "classifier",
            LinearSVC(
                random_state=42,
            ),
        ),
    ]
)

arabic_pipeline.fit(
    arabic_texts,
    arabic_labels,
)

arabic_pipeline.predict(
    [
        "لا يمكنني فتح الحساب",
        "أحتاج الفاتورة",
        "البرنامج يتعطل",
    ]
)

This small example demonstrates the workflow only. Reliable Arabic evaluation
requires larger, variety-aware datasets.

# 20. Reproducibility and Deployment

Record:

- data version;
- split seed;
- preprocessing;
- vectorizer parameters;
- model hyperparameters;
- library versions;
- evaluation metrics;
- label definitions.

Production monitoring should track:

- missing or malformed inputs;
- class distribution changes;
- confidence shifts;
- error reports;
- latency;
- drift.

In [ ]:
import sklearn

reproducibility_info = pd.Series(
    {
        "random_state": 42,
        "vectorizer": "TF-IDF with word 1-2 grams",
        "models": "MultinomialNB, LogisticRegression, LinearSVC",
        "scikit-learn_version": sklearn.__version__,
    },
    name="Experiment information",
)

reproducibility_info

# 21. Knowledge Check

1. What is supervised text classification?
2. Why should raw text be split before vectorizer fitting?
3. What assumption does Naive Bayes make?
4. What does Naive Bayes smoothing control?
5. What does Logistic Regression learn?
6. What is the margin idea in a linear SVM?
7. How do precision and recall differ?
8. Why is macro F1 useful?
9. What does a confusion matrix show?
10. Why is cross-validation useful?
11. Why can accuracy mislead on imbalanced data?
12. What does `class_weight="balanced"` do?
13. Why must hyperparameter tuning avoid the final test set?
14. How can linear feature coefficients support interpretation?
15. Which issues affect Arabic text classification?

# 22. Exercises

## Exercise 1 — Binary Classification

Build a spam classifier using TF-IDF and Logistic Regression.

## Exercise 2 — Model Comparison

Compare Naive Bayes, Logistic Regression, and Linear SVM using the same folds.

## Exercise 3 — Metrics

Calculate accuracy, macro precision, macro recall, and macro F1 manually from a
confusion matrix.

## Exercise 4 — Class Imbalance

Create an imbalanced dataset and compare ordinary Logistic Regression with
class-weighted Logistic Regression.

## Exercise 5 — Hyperparameter Tuning

Tune n-gram range, `min_df`, `alpha`, and `C`.

## Exercise 6 — Feature Interpretation

Display the top ten features for each class.

## Exercise 7 — Error Analysis

Create an error table containing text, actual label, predicted label, and
confidence.

## Exercise 8 — Arabic Classification

Compare Arabic word TF-IDF and character TF-IDF.

## Challenge Exercises

1. Add probability calibration to a linear SVM.
2. Compare one-vs-rest and multinomial strategies.
3. Perform nested cross-validation for model selection.
4. Build a reusable training and evaluation function.
5. Add drift monitoring for class distribution and vocabulary coverage.

# 23. Summary and Next Lesson

In this lesson:

- supervised text classification mapped labeled text to classes;
- pipelines prevented vectorizer leakage;
- Multinomial Naive Bayes provided a fast probabilistic baseline;
- Logistic Regression provided linear probabilities and interpretable
  coefficients;
- linear SVMs provided strong maximum-margin baselines;
- precision, recall, F1, and macro F1 evaluated class performance;
- confusion matrices exposed specific class confusions;
- stratified cross-validation measured score stability;
- class weights supported imbalanced learning;
- GridSearchCV tuned vectorizer and model hyperparameters safely;
- feature inspection and error analysis explained model behavior;
- multilingual and Arabic classification required language-aware features and
  evaluation.

## Next Lesson

**Lesson 17: Information Retrieval and Document Similarity** introduces
indexing, query-document matching, cosine similarity, ranking, precision,
recall, Mean Reciprocal Rank, and classical search pipelines.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Manning, C. D., Raghavan, P., & Schütze, H. *Introduction to Information Retrieval*.
- scikit-learn text classification documentation.
- classical Naive Bayes, Logistic Regression, and SVM literature.
- Arabic text-classification literature.